## ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Project 3: NLP Classification: Subreddit Pepsi vs Coca-Cola | Part 3: Vectorizer Peformance

---

[README](../README.md) | [Part 1: EDA](01_EDA.ipynb)  | [Part 2: Vectorizer](02_Vectorizer.ipynb) | **Part 3: Vectorizer Performance**

---

### Introduction
In this section, we will analyze the results of various parameter combinations for three models—Naive Bayes and XGBoost—from [Part 2: Vectorizer](02_Vectorizer.ipynb). The primary objective of this analysis is to evaluate how different parameter configurations impact each model's accuracy score. By systematically examining the performance across these models, we aim to identify the key parameters that drive model accuracy and understand the trade-offs associated with different settings. This analysis will provide valuable insights into optimizing model performance and selecting the most effective hyperparameter configurations.

### Import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

### Data Preparation
Some parameter combinations result in `NaN` accuracy scores during cross-validation, often due to issues like data imbalance, computational errors (e.g., division by zero), or small datasets that prevent the model from learning effectively. To ensure a reliable analysis, we will exclude these `NaN` results and focus only on the valid parameter combinations that produce meaningful accuracy scores.

For this analysis, we will use the following columns: 
- `param_classifier`: The classifier model used (e.g., Naive Bayes, AdaBoost and XGBoost)
- `param_vectorizer`: The type of vectorizer applied (e.g., CountVectorizer, TfidfVectorizer)
- `param_vectorizer__tokenizer`: The tokenizer function used for text tokenization 
- `param_vectorizer__max_features`: The maximum number of features for the vectorizer; possible values include 3000, 5000, or None
- `param_vectorizer__ngram_range`: The range of n-grams used for tokenization; options include unigrams or both unigrams and bigrams
- `param_vectorizer__min_df`: The minimum document frequency for terms to be considered in the vectorizer; options are 2 or 3
- `param_vectorizer__max_df`: The maximum document frequency for terms to be considered in the vectorizer; options are 0.8 or 0.9
- `mean_train_accuracy`: The mean accuracy score calculated from the training data
- `mean_test_accuracy`: The mean accuracy score calculated from the testing data

In [2]:
df1 = pd.read_csv('../data/cv_results_nb.csv')                       # Load Data Naive Bayes Performance
df2 = pd.read_csv('../data/cv_results_xgb.csv')                      # Load Data XGBoost Performance 
df3 = pd.read_csv('../data/cv_results_ada.csv')                      # Load Data AdaBoost Performance 
# Check shape
df1.shape, df2.shape, df3.shape                                                       

((288, 73), (288, 73), (288, 73))

In [3]:
# Concatenate 3 DataFrames into single DataFrame.
# Note that all DataFrames have same structures.
df = pd.concat([df1, df2, df3], axis = 0, ignore_index = True)

# Check
df.head()

,Unnamed: 0,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_vectorizer,param_vectorizer__max_df,param_vectorizer__max_features,param_vectorizer__min_df,...,mean_test_f1,std_test_f1,rank_test_f1,split0_train_f1,split1_train_f1,split2_train_f1,split3_train_f1,split4_train_f1,mean_train_f1,std_train_f1
0,0,6.482064,0.374552,2.387187,0.530890,MultinomialNB(),CountVectorizer(token_pattern=None),0.8,3000.0,2,...,0.656966,0.020020,186,0.808511,0.802022,0.809483,0.790186,0.801695,0.802379,0.006890
1,1,6.820330,0.470872,3.450289,1.457087,MultinomialNB(),CountVectorizer(token_pattern=None),0.8,3000.0,2,...,0.668008,0.027307,147,0.821963,0.812242,0.825581,0.805042,0.814381,0.815842,0.007267
2,2,15.779505,1.088593,4.674823,1.028505,MultinomialNB(),CountVectorizer(token_pattern=None),0.8,3000.0,2,...,0.652557,0.014684,215,0.801370,0.775300,0.798291,0.792230,0.792929,0.792024,0.009024
3,3,10.980268,3.065498,3.471096,1.715701,MultinomialNB(),CountVectorizer(token_pattern=None),0.8,3000.0,2,...,0.668405,0.025770,139,0.811400,0.787521,0.816807,0.797320,0.811305,0.804871,0.010812
4,4,20.976396,2.336415,0.463699,0.927397,MultinomialNB(),CountVectorizer(token_pattern=None),0.8,3000.0,2,...,NaN,NaN,287,0.799658,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Focus Columns
columns = ['param_classifier' 
           , 'param_vectorizer'
           , 'param_vectorizer__tokenizer' 
           , 'param_vectorizer__max_features' 
           , 'param_vectorizer__ngram_range' 
           , 'param_vectorizer__min_df' 
           , 'param_vectorizer__max_df'
           , 'mean_train_accuracy' 
           , 'mean_test_accuracy'
          ]

df = df[columns]

In [5]:
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending = False).head()

param_vectorizer__max_features    288
mean_train_accuracy                 4
mean_test_accuracy                  4
dtype: int64

In [6]:
# Map Classifiers Name
def map_classifier_name(classifier_name):
    if 'AdaBoostClassifier' in classifier_name:
        return 'AdaBoost'
    elif 'XGBClassifier' in classifier_name:
        return 'XGBoost'
    elif 'MultinomialNB' in classifier_name:
        return 'Naive Bayes'
    else:
        return 'Unknown'  # In case there are any other classifiers

df['param_classifier'] = df['param_classifier'].apply(map_classifier_name)

In [7]:
# Tokenizer name
tokenizers = ['lem_only', 'lem_stop', 'lower_only', 'lower_stop', 'stem_only', 'stem_stop']

for word in tokenizers:
    df['param_vectorizer__tokenizer'] = np.where(
        df['param_vectorizer__tokenizer'].str.contains(word, na=False), 
        word, 
        df['param_vectorizer__tokenizer']
    )

In [8]:
df['param_vectorizer__tokenizer'].value_counts()

param_vectorizer__tokenizer
lower_only    144
lower_stop    144
stem_only     144
stem_stop     144
lem_only      144
lem_stop      144
Name: count, dtype: int64

In [9]:
# Drop rows that mean_train_accuracy or mean_test_accuracy is NaN
df = df.dropna(subset=['mean_train_accuracy','mean_test_accuracy'])

### Vectorizer and Tokenizer Performance
- We compare performance by analyzing the effect of each parameter while keeping the classifiers at their default settings.
- Key findings include:
    - Across all models, TfidfVectorizer outperforms CountVectorizer in training accuracy, but both perform equally in testing accuracy.
    - For Naive Bayes, the choice of tokenizer has minimal impact on the scores. In XGBoost, retaining stopwords results in higher training accuracy, while in AdaBoost, using lemmatization or stemming improves the scores.
- Recommedation
    - For prioritizing testing accuracy and generalization: Naive Bayes with either vectorizer is the best choice.
    - For aiming at maximum training accuracy with strategies to mitigate overfitting: XGBoost with TfidfVectorizer is recommended.
    - For balanced performance or interpretability: AdaBoost with TfidfVectorizer is the ideal option.

In [10]:
df.groupby(['param_classifier','param_vectorizer'])\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

mean_train_accuracy  \
param_classifier param_vectorizer                                           
AdaBoost         CountVectorizer(token_pattern=None)                   64   
                 TfidfVectorizer(token_pattern=None)                   65   
Naive Bayes      CountVectorizer(token_pattern=None)                   80   
                 TfidfVectorizer(token_pattern=None)                   87   
XGBoost          CountVectorizer(token_pattern=None)                   89   
                 TfidfVectorizer(token_pattern=None)                   93   

                                                      mean_test_accuracy  
param_classifier param_vectorizer                                         
AdaBoost         CountVectorizer(token_pattern=None)                  62  
                 TfidfVectorizer(token_pattern=None)                  62  
Naive Bayes      CountVectorizer(token_pattern=None)                  67  
                 TfidfVectorizer(token_pattern=None)                  67  
XGBoost          CountVectorizer(token_pattern=None)                  66  
                 TfidfVectorizer(token_pattern=None)                  64

In [11]:
df.groupby(['param_classifier','param_vectorizer','param_vectorizer__tokenizer'])\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

mean_train_accuracy  \
param_classifier param_vectorizer                    param_vectorizer__tokenizer                        
AdaBoost         CountVectorizer(token_pattern=None) lem_only                                      64   
                                                     lem_stop                                      64   
                                                     lower_only                                    62   
                                                     lower_stop                                    62   
                                                     stem_only                                     65   
                                                     stem_stop                                     65   
                 TfidfVectorizer(token_pattern=None) lem_only                                      64   
                                                     lem_stop                                      64   
                                                     lower_only                                    64   
                                                     lower_stop                                    64   
                                                     stem_only                                     67   
                                                     stem_stop                                     66   
Naive Bayes      CountVectorizer(token_pattern=None) lem_only                                      80   
                                                     lem_stop                                      80   
                                                     lower_only                                    81   
                                                     lower_stop                                    81   
                                                     stem_only                                     80   
                                                     stem_stop                                     80   
                 TfidfVectorizer(token_pattern=None) lem_only                                      88   
                                                     lem_stop                                      86   
                                                     lower_only                                    88   
                                                     lower_stop                                    87   
                                                     stem_only                                     87   
                                                     stem_stop                                     86   
XGBoost          CountVectorizer(token_pattern=None) lem_only                                      92   
                                                     lem_stop                                      87   
                                                     lower_only                                    91   
                                                     lower_stop                                    87   
                                                     stem_only                                     92   
                                                     stem_stop                                     87   
                 TfidfVectorizer(token_pattern=None) lem_only                                      95   
                                                     lem_stop                                      91   
                                                     lower_only                                    95   
                                                     lower_stop                                    90   
                                                     stem_only                                     96   
                                                     stem_stop                                     91   

                                                                                  mean_test

### Tuning Vectorizer for Naive Bayes

In [12]:
# df.groupby(['param_vectorizer__max_features'], dropna = False)[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

In [13]:
# df.groupby(['param_vectorizer__ngram_range'])[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

In [14]:
# df.groupby(['param_vectorizer__min_df'])[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

In [15]:
# df.groupby(['param_vectorizer__max_df'])[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

In [24]:
df[(df['param_classifier'] == 'Naive Bayes') & (df['param_vectorizer'] == 'TfidfVectorizer(token_pattern=None)')]\
.groupby(['param_vectorizer__max_features'], dropna = False)\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

,mean_train_accuracy,mean_test_accuracy
param_vectorizer__max_features,,
3000.0,86,67
5000.0,87,67
NaN,87,67


In [25]:
df[(df['param_classifier'] == 'Naive Bayes') & (df['param_vectorizer'] == 'TfidfVectorizer(token_pattern=None)')]\
.groupby(['param_vectorizer__ngram_range'], dropna = False)\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

,mean_train_accuracy,mean_test_accuracy
param_vectorizer__ngram_range,,
"(1, 1)",86,68
"(1, 2)",88,67


In [26]:
df[(df['param_classifier'] == 'Naive Bayes') & (df['param_vectorizer'] == 'TfidfVectorizer(token_pattern=None)')]\
.groupby(['param_vectorizer__min_df'], dropna = False)\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

,mean_train_accuracy,mean_test_accuracy
param_vectorizer__min_df,,
2,88,68
3,85,67


In [27]:
df[(df['param_classifier'] == 'Naive Bayes') & (df['param_vectorizer'] == 'TfidfVectorizer(token_pattern=None)')]\
.groupby(['param_vectorizer__max_df'], dropna = False)\
[['mean_train_accuracy', 'mean_test_accuracy']].mean().mul(100).astype(int)

,mean_train_accuracy,mean_test_accuracy
param_vectorizer__max_df,,
0.8,87,67
0.9,87,67
